# 04 - The Training Loop

**Goal:** Master the standard PyTorch training pattern.

---

## The Pattern

Every PyTorch training loop follows this structure:

```python
for epoch in range(num_epochs):
    for batch in dataloader:
        # 1. Forward pass
        predictions = model(inputs)
        loss = criterion(predictions, targets)
        
        # 2. Backward pass
        optimizer.zero_grad()   # Clear old gradients
        loss.backward()         # Compute gradients
        
        # 3. Update weights
        optimizer.step()        # Apply gradients
```

Let's understand each component.

## Loss Functions (Criterion)

The loss function measures how wrong the predictions are.

| Task | Loss Function | PyTorch |
|------|--------------|--------|
| Classification | Cross-Entropy | `nn.CrossEntropyLoss()` |
| Binary classification | Binary Cross-Entropy | `nn.BCEWithLogitsLoss()` |
| Regression | Mean Squared Error | `nn.MSELoss()` |
| Segmentation | Dice Loss + Cross-Entropy | Custom (MONAI has it) |

In [ ]:
import torch
import torch.nn as nn

# Cross-Entropy Loss for classification
criterion = nn.CrossEntropyLoss()

# Example: 3 samples, 5 classes
predictions = torch.randn(3, 5)  # Raw model output (logits)
targets = torch.tensor([0, 3, 2])  # True class indices

loss = criterion(predictions, targets)
print(f"Predictions shape: {predictions.shape}")
print(f"Targets: {targets}")
print(f"Loss: {loss.item():.4f}")

In [ ]:
# MSE Loss for regression
mse = nn.MSELoss()

predictions = torch.tensor([2.5, 0.0, 2.0])
targets = torch.tensor([3.0, -0.5, 2.0])

loss = mse(predictions, targets)
print(f"Predictions: {predictions}")
print(f"Targets: {targets}")
print(f"MSE Loss: {loss.item():.4f}")

# Manual calculation: mean of squared differences
manual = ((predictions - targets) ** 2).mean()
print(f"Manual calculation: {manual.item():.4f}")

## Optimizers

Optimizers update the model parameters based on gradients.

| Optimizer | Description | When to use |
|-----------|-------------|------------|
| SGD | Basic gradient descent | Simple, need to tune LR |
| Adam | Adaptive learning rates | Most common, good default |
| AdamW | Adam with weight decay | Modern best practice |

In [ ]:
import torch.optim as optim

# Create a simple model
model = nn.Linear(10, 5)

# SGD optimizer
optimizer_sgd = optim.SGD(model.parameters(), lr=0.01)

# Adam optimizer (most common)
optimizer_adam = optim.Adam(model.parameters(), lr=0.001)

# AdamW (recommended for transformers)
optimizer_adamw = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

print(f"SGD parameters: {optimizer_sgd.defaults}")
print(f"Adam parameters: {optimizer_adam.defaults}")

## DataLoader

DataLoader batches your data and handles shuffling:

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Create fake dataset
X = torch.randn(100, 10)  # 100 samples, 10 features
y = torch.randint(0, 5, (100,))  # 100 labels, 5 classes

# Wrap in TensorDataset
dataset = TensorDataset(X, y)

# Create DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=16,      # Samples per batch
    shuffle=True,       # Randomize order each epoch
    num_workers=0       # Parallel data loading (0 for simple cases)
)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(dataloader)}")

# Iterate
for i, (batch_x, batch_y) in enumerate(dataloader):
    print(f"Batch {i}: X shape = {batch_x.shape}, y shape = {batch_y.shape}")
    if i >= 2:
        print("...")
        break

## Complete Training Loop

In [ ]:
import matplotlib.pyplot as plt

# Setup
# -----

# Model
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 5)
)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Data (same as before)
X = torch.randn(100, 10)
y = torch.randint(0, 5, (100,))
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print("Setup complete!")

In [ ]:
# Training loop
# -------------

num_epochs = 50
epoch_losses = []

model.train()  # Set to training mode

for epoch in range(num_epochs):
    epoch_loss = 0.0
    
    for batch_x, batch_y in dataloader:
        # Forward pass
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        
        # Backward pass
        optimizer.zero_grad()  # IMPORTANT: Clear gradients
        loss.backward()        # Compute gradients
        
        # Update weights
        optimizer.step()       # Apply gradients
        
        epoch_loss += loss.item()
    
    # Average loss for this epoch
    avg_loss = epoch_loss / len(dataloader)
    epoch_losses.append(avg_loss)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}: Loss = {avg_loss:.4f}")

print("\nTraining complete!")

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(epoch_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.grid(True, alpha=0.3)
plt.show()

## Evaluation

After training, evaluate on held-out data:

In [ ]:
# Create test data
X_test = torch.randn(20, 10)
y_test = torch.randint(0, 5, (20,))

# Evaluation
model.eval()  # Set to evaluation mode

with torch.no_grad():  # No gradients needed
    predictions = model(X_test)
    predicted_classes = predictions.argmax(dim=1)
    
    # Accuracy
    correct = (predicted_classes == y_test).sum().item()
    accuracy = correct / len(y_test)
    
print(f"Predictions: {predicted_classes.tolist()}")
print(f"True labels: {y_test.tolist()}")
print(f"Accuracy: {accuracy:.2%}")

## Training with Validation

Better practice: check performance on validation set each epoch:

In [ ]:
# Split data into train and validation
X = torch.randn(200, 10)
y = torch.randint(0, 5, (200,))

# 80% train, 20% validation
train_X, val_X = X[:160], X[160:]
train_y, val_y = y[:160], y[160:]

train_loader = DataLoader(TensorDataset(train_X, train_y), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(val_X, val_y), batch_size=16)

print(f"Train: {len(train_X)}, Val: {len(val_X)}")

In [ ]:
# Training with validation

model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 5)
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

train_losses = []
val_losses = []

for epoch in range(50):
    # Training
    model.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    train_losses.append(train_loss / len(train_loader))
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item()
    val_losses.append(val_loss / len(val_loader))
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}: Train Loss = {train_losses[-1]:.4f}, Val Loss = {val_losses[-1]:.4f}")

In [ ]:
# Plot both
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("If validation loss increases while train decreases = overfitting!")

## GPU Training

In [ ]:
# Only change: move model and data to device

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 5)
).to(device)  # <-- Move model to GPU

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop with GPU
for epoch in range(10):
    for batch_x, batch_y in train_loader:
        # Move data to GPU
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    if epoch % 3 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

## Summary: The Template

```python
# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MyModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training
for epoch in range(num_epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    with torch.no_grad():
        # Evaluate on val_loader...

# Save
torch.save(model.state_dict(), 'model.pth')
```

**Next:** Apply everything to MNIST - a real working classifier!